# Pixel Analysis — Hashed PII Leaks

This notebook detects **personal data exfiltration** through tracking pixels on pharmacy websites.

During data collection, unique persona details (email, name, phone, etc.) were entered into checkout forms. This notebook searches for SHA-256 hashes of those details in request URLs and POST bodies.

## Setup and data load

In [ ]:
import pandas as pd
import seaborn as sns
from utils import parse_har_files_to_dataframe
from common import BASE_HAR_DIR


sns.set_theme(style="whitegrid", context="paper")
pd.set_option('display.max_colwidth', 128)

df = parse_har_files_to_dataframe(BASE_HAR_DIR, test=False, use_cache=True)

### Search for hashed persona details
- For each persona field (email, name, phone, etc.), compute its SHA-256 hash and search for it in request URLs and POST data
- A match indicates the site or a third party is exfiltrating hashed PII via tracking pixels

In [ ]:
from common import PERSONA_DETAILS

from hashlib import sha256
DISPLAY_COLS = ['exp_mode', 'consent_mode', 'country', 'site_domain', 'req_domain']

for persona_name, persona_details in PERSONA_DETAILS.items():
    for persona_field, value in persona_details.items():
        print(persona_name, persona_field, value)
        hashed_field_value = sha256(value.encode()).hexdigest()
        filtered = df[df.post_data.str.contains(hashed_field_value) |
                    df.url.str.contains(hashed_field_value)]
        if not filtered.empty:
            display(filtered.drop_duplicates(DISPLAY_COLS)[DISPLAY_COLS + ['url']])

### Summary: hashed PII leaks

In [ ]:
from hashlib import sha256 as _sha256

leak_records = []
for persona_name, persona_details in PERSONA_DETAILS.items():
    for persona_field, value in persona_details.items():
        h = _sha256(value.encode()).hexdigest()
        hits = df[df.post_data.str.contains(h) | df.url.str.contains(h)]
        if not hits.empty:
            for _, row in hits.drop_duplicates(DISPLAY_COLS).iterrows():
                leak_records.append({
                    'persona': persona_name,
                    'field': persona_field,
                    'exp_mode': row['exp_mode'],
                    'consent_mode': row['consent_mode'],
                    'site_domain': row['site_domain'],
                    'receiver': row['req_domain'],
                })

df_leaks = pd.DataFrame(leak_records)
print(f"Total hashed PII leak instances: {len(df_leaks)}")

if df_leaks.empty:
    print("\nNo hashed PII leaks detected (are real persona details configured in common.py?)")
else:
    print(f"Sites leaking hashed PII:        {df_leaks['site_domain'].nunique()}")
    print(f"Receiving domains:                {df_leaks['receiver'].nunique()}")
    print(f"Leaked field types:               {df_leaks['field'].nunique()}")
    print()
    pivot = df_leaks.pivot_table(
        index=['site_domain', 'receiver'],
        columns='consent_mode',
        values='field',
        aggfunc=lambda x: ', '.join(sorted(set(x)))
    ).fillna('—')
    display(pivot)

### Facebook Pixel Automated Advanced Matching
- udff[em] = sha256 of email
- udff[ph] = sha256 of phone
...

In [ ]:
df[(df.req_domain == 'facebook.com') &
        df.param_names.astype(str).str.contains("udff[", regex=False)].\
    drop_duplicates(subset=['site_domain', 'country', 'consent_mode'])